<a href="https://colab.research.google.com/github/KaraIbr/SmartParkingGU/blob/main/notebooks/02_base_model_cnrpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Modelo Base: SmartParkingCNN

**Fase:** Arquitectura y Entrenamiento Base (30% de la rubrica)

**Objetivo:**
- Construir una red neuronal convolucional desde cero
- Entrenar el modelo base
- Reportar metricas iniciales
- Visualizar curvas de perdida y precision

**Dataset:** CNRPark-Patches-150x150

## 1. Configuracion e Imports

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from tqdm import tqdm

# Agregar directorio raiz al path para importar scripts/
sys.path.insert(0, os.path.abspath('..'))

from scripts import (
    ParkingDataset,
    SmartParkingCNN,
    get_model_info,
    train_model,
    load_checkpoint,
    evaluate_model,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_pr_curve,
    plot_training_curves,
    print_metrics,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Cargar Dataset Preprocesado

In [ ]:
# Configuracion
DATASET_ROOT = './dataset'
BATCH_SIZE = 32
IMG_SIZE = (150, 150)

# Transformaciones para entrenamiento (sin augmentation en este paso)
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transformaciones para validacion/test
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Crear datasets usando ParkingDataset de scripts/
train_dataset = ParkingDataset(DATASET_ROOT, split='entrenamiento', transform=train_transform)
val_dataset = ParkingDataset(DATASET_ROOT, split='validacion', transform=val_transform)
test_dataset = ParkingDataset(DATASET_ROOT, split='test', transform=val_transform)

# Crear dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Dataset cargado correctamente:")
print(f"  - Entrenamiento: {len(train_dataset)} imagenes")
print(f"  - Validacion:    {len(val_dataset)} imagenes")
print(f"  - Prueba:        {len(test_dataset)} imagenes")
print(f"  - Total:         {len(train_dataset) + len(val_dataset) + len(test_dataset)} imagenes")

In [ ]:
# Mostrar muestras del dataset
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Muestras del Dataset - Entrenamiento', fontsize=14, fontweight='bold')

class_names = ['Ocupado', 'Libre']
for idx, class_name in enumerate(['ocupado', 'no_ocupado']):
    class_dir = os.path.join(DATASET_ROOT, 'entrenamiento', class_name)
    imgs = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
    for j, img_name in enumerate(imgs):
        img_path = os.path.join(class_dir, img_name)
        img = plt.imread(img_path)
        axes[idx, j].imshow(img)
        axes[idx, j].set_title(f'{class_names[idx]}', fontsize=10)
        axes[idx, j].axis('off')

plt.tight_layout()
plt.savefig('reports/figures/muestras_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribucion de clases
train_counts = {}
for cls in ['ocupado', 'no_ocupado']:
    path = os.path.join(DATASET_ROOT, 'entrenamiento', cls)
    train_counts[cls] = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

plt.figure(figsize=(8, 5))
bars = plt.bar(['Ocupado', 'Libre'], [train_counts['ocupado'], train_counts['no_ocupado']],
               color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.5)
plt.title('Distribucion de Clases - Entrenamiento', fontsize=14, fontweight='bold')
plt.ylabel('Numero de Imagenes')
for bar, count in zip(bars, [train_counts['ocupado'], train_counts['no_ocupado']]):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             f'{count}', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, max(train_counts.values()) * 1.15)
plt.tight_layout()
plt.savefig('reports/figures/distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Definir Arquitectura CNN

In [ ]:
# Configuracion del dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Crear modelo
model = SmartParkingCNN(num_classes=2, dropout_rate=0.5)
model = model.to(device)

# Mostrar informacion del modelo
model_info = get_model_info(model)
print(f"\nArquitectura: {model_info['arquitectura']}")
print(f"Capas convolucionales: {model_info['capas_convolucionales']}")
print(f"Capas FC: {model_info['capas_fc']}")
print(f"Parametros totales: {model_info['parametros_miles']}")

# Mostrar resumen de la arquitectura
print(f"\nResumen de la arquitectura:")
print(model)

## 4. Entrenamiento Base

Hiperparametros iniciales segun el plan.md:
- Optimizer: Adam
- Learning Rate: 1e-3
- Batch Size: 32
- Epochs: 60
- EarlyStopping: patience=10
- Loss: CrossEntropyLoss

In [ ]:
# Hiperparametros base
BASE_CONFIG = {
    'learning_rate': 1e-3,
    'weight_decay': 0,
    'dropout_rate': 0.5,
    'batch_size': 32,
    'num_epochs': 60,
    'patience': 10
}

print("Hiperparametros base:")
for key, value in BASE_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Entrenar modelo base
history_base = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=BASE_CONFIG['num_epochs'],
    learning_rate=BASE_CONFIG['learning_rate'],
    weight_decay=BASE_CONFIG['weight_decay'],
    device=device,
    save_dir='experiments/cnrpark/base_model',
    patience=BASE_CONFIG['patience']
)

## 5. Curvas de Entrenamiento

In [ ]:
# Visualizar curvas de loss y accuracy
plot_training_curves(
    history_base,
    save_path='reports/figures/training_curves_base.png',
    title='Curvas de Entrenamiento - Modelo Base'
)
plt.show()

## 6. Evaluacion del Modelo Base

Metricas requeridas:
- Accuracy
- Precision
- Recall
- F1-Score
- Matriz de confusion
- ROC-AUC curve
- PR curve

In [ ]:
# Cargar mejor modelo
model_base = SmartParkingCNN(num_classes=2, dropout_rate=BASE_CONFIG['dropout_rate'])
model_base, checkpoint = load_checkpoint(
    model_base,
    'experiments/cnrpark/base_model/best_model.pt'
)
model_base = model_base.to(device)

print(f"Modelo cargado desde epoca {checkpoint.get('epoch', 'N/A')}")
print(f"Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}")
print(f"Val Acc: {checkpoint.get('val_acc', 'N/A'):.2f}%")

In [ ]:
# Evaluar en conjunto de prueba
metrics_base, preds_base, labels_base, probs_base = evaluate_model(
    model_base,
    test_loader,
    device,
    class_names=['Ocupado', 'Libre']
)

# Mostrar metricas
print_metrics(metrics_base, "METRICAS BASE - TEST SET")

### 6.1 Matriz de Confusion

In [ ]:
# Matriz de confusion
cm_base = plot_confusion_matrix(
    labels_base,
    preds_base,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/confusion_matrix_base.png',
    title='Matriz de Confusion - Modelo Base'
)
plt.show()

### 6.2 ROC Curve

In [ ]:
# ROC Curve
fpr_base, tpr_base, roc_auc_base = plot_roc_curve(
    labels_base,
    probs_base,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/roc_curve_base.png',
    title='ROC Curve - Modelo Base'
)
plt.show()
print(f"\nROC-AUC Score: {roc_auc_base:.4f}")

### 6.3 Precision-Recall Curve

In [ ]:
# PR Curve
precision_pr_base, recall_pr_base, pr_auc_base = plot_pr_curve(
    labels_base,
    probs_base,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/pr_curve_base.png',
    title='Precision-Recall Curve - Modelo Base'
)
plt.show()
print(f"\nPR-AUC Score: {pr_auc_base:.4f}")

## 7. K-Fold Cross Validation

Ejecutar 3-fold stratified CV para evaluar variabilidad del modelo.

In [ ]:
def kfold_cross_validation(dataset, n_splits=3, num_epochs=20, device='cuda'):
    """
    Ejecuta K-Fold Cross Validation.
    
    Args:
        dataset: Dataset completo
        n_splits: Numero de folds
        num_epochs: Epocas por fold
        device: 'cuda' o 'cpu'
    
    Returns:
        results: Lista de metricas por fold
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Obtener indices
    indices = list(range(len(dataset)))
    labels = [dataset.samples[i]['label'] for i in indices]
    
    results = []
    
    print(f"\n{'='*60}")
    print(f"K-FOLD CROSS VALIDATION (k={n_splits})")
    print(f"{'='*60}\n")
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(indices), 1):
        print(f"\n--- Fold {fold}/{n_splits} ---")
        
        # Crear subdatasets
        train_subset = torch.utils.data.Subset(dataset, train_idx)
        val_subset = torch.utils.data.Subset(dataset, val_idx)
        
        # Crear dataloaders
        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
        
        # Crear modelo nuevo
        model = SmartParkingCNN(num_classes=2, dropout_rate=0.5).to(device)
        
        # Entrenar
        history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            learning_rate=1e-3,
            device=device,
            save_dir=f'experiments/cnrpark/kfold_fold_{fold}',
            patience=7
        )
        
        # Evaluar en validation
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels_batch in val_loader:
                images = images.to(device)
                labels_batch = labels_batch.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels_batch.cpu().numpy())
        
        # Calcular metricas
        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
        
        fold_metrics = {
            'fold': fold,
            'accuracy': accuracy_score(all_labels, all_preds),
            'precision': precision_score(all_labels, all_preds, average='macro'),
            'recall': recall_score(all_labels, all_preds, average='macro'),
            'f1_score': f1_score(all_labels, all_preds, average='macro')
        }
        results.append(fold_metrics)
        
        print(f"Fold {fold}: Acc={fold_metrics['accuracy']*100:.2f}%, "
              f"F1={fold_metrics['f1_score']*100:.2f}%")
    
    # Resumen
    print(f"\n{'='*60}")
    print("RESUMEN K-FOLD CV")
    print(f"{'='*60}")
    
    for metric_name in ['accuracy', 'precision', 'recall', 'f1_score']:
        values = [r[metric_name] for r in results]
        print(f"{metric_name.upper():>10}: {np.mean(values)*100:.2f}% (+/- {np.std(values)*100:.2f}%)")
    
    print(f"{'='*60}")
    
    return results

In [ ]:
# Crear dataset completo para K-Fold
full_dataset = ParkingDataset(DATASET_ROOT, split='entrenamiento', transform=val_transform)

# Ejecutar K-Fold CV
kfold_results = kfold_cross_validation(
    full_dataset,
    n_splits=3,
    num_epochs=20,
    device=device
)

## 8. Resumen del Modelo Base

In [ ]:
# Resumen final
print("=" * 60)
print("RESUMEN - MODELO BASE")
print("=" * 60)
print(f"\nHiperparametros:")
for key, value in BASE_CONFIG.items():
    print(f"  {key}: {value}")

print(f"\nMetricas en Test Set:")
print(f"  Accuracy:  {metrics_base['accuracy']*100:.2f}%")
print(f"  Precision: {metrics_base['precision_macro']*100:.2f}%")
print(f"  Recall:    {metrics_base['recall_macro']*100:.2f}%")
print(f"  F1-Score:  {metrics_base['f1_macro']*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_base:.4f}")

print(f"\nK-Fold CV (k=3):")
for metric_name in ['accuracy', 'f1_score']:
    values = [r[metric_name] for r in kfold_results]
    print(f"  {metric_name.upper()}: {np.mean(values)*100:.2f}% (+/- {np.std(values)*100:.2f}%)")

print(f"\nModelo guardado en: experiments/cnrpark/base_model/")
print("=" * 60)

In [ ]:
# Guardar metricas para el notebook 03 (fine-tuning)
os.makedirs('experiments/cnrpark/base_model', exist_ok=True)

base_metrics_save = {
    'accuracy': float(metrics_base['accuracy']),
    'precision_macro': float(metrics_base['precision_macro']),
    'recall_macro': float(metrics_base['recall_macro']),
    'f1_macro': float(metrics_base['f1_macro']),
    'roc_auc': float(roc_auc_base),
}

with open('experiments/cnrpark/base_model/metrics.json', 'w') as f:
    json.dump(base_metrics_save, f, indent=2)

print('Metricas guardadas en experiments/cnrpark/base_model/metrics.json')

## 9. Siguiente Paso

El modelo base ha sido entrenado y evaluado. Ahora procederemos a:

1. **Fine-Tuning y Optimizacion** en `03_finetune_cnrpark.ipynb`
2. Busqueda de mejores hiperparametros
3. Mejora de metricas sobre el modelo base